# 📖 Notebook 1: Operational Transformation Basics

When two people edit the same document at the same time, their edits can **conflict**. Operational Transformation (OT) is the algorithm that resolves these conflicts so everyone sees the same final document.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why naive concurrent editing breaks documents
- What an "operation" is (insert/delete at a position)
- How OT transforms operations to preserve intent
- Why Google Docs chose OT over other approaches

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/google-docs
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `googledocs_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import json

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "googledocs_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

# Test connection
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

## 🤔 The Problem: Concurrent Edits Break Documents

Imagine Alice and Bob are both editing the word `Hello!`:

```
Starting document: "Hello!"
                    012345

Alice wants to add ", world" after "Hello"  →  INSERT(5, ", world")
Bob wants to delete the "!"                  →  DELETE(5, 1)
```

If we just apply both edits naively, the order they arrive matters:

- **Alice first, then Bob**: `Hello, world!` → delete at 5 → `Hello world!` ❌ (deleted the comma!)
- **Bob first, then Alice**: `Hello` → insert at 5 → `Hello, world` ✅

**Neither order gives us what both users intended!** Alice wanted `Hello, world` and Bob wanted the `!` removed.

The correct result should be: **`Hello, world`**

This is exactly the problem OT solves.

In [ ]:
# Let's see the problem in code

def apply_insert(text, position, content):
    """Insert `content` at `position` in the text."""
    return text[:position] + content + text[position:]

def apply_delete(text, position, length):
    """Delete `length` characters starting at `position`."""
    return text[:position] + text[position + length:]

original = "Hello!"
print(f"Original document: '{original}'")
print(f"Positions:          {''.join(str(i) for i in range(len(original)))}")
print()

# Alice's edit: INSERT ", world" at position 5
alice_op = {"type": "insert", "position": 5, "content": ", world"}

# Bob's edit: DELETE 1 char at position 5 (the "!")
bob_op = {"type": "delete", "position": 5, "length": 1}

print("Alice's intent: INSERT(5, ', world') — add ', world' after 'Hello'")
print("Bob's intent:   DELETE(5, 1) — remove the '!'")
print()

# Scenario 1: Alice first, then Bob (WRONG)
after_alice = apply_insert(original, 5, ", world")
after_both_1 = apply_delete(after_alice, 5, 1)  # Bob's DELETE(5,1) hits the comma!
print(f"Order 1 (Alice→Bob): '{original}' → '{after_alice}' → '{after_both_1}'")
print(f"  ❌ Bob deleted the comma instead of the '!'")
print()

# Scenario 2: Bob first, then Alice (ALSO WRONG — missing the '!' deletion intent)
after_bob = apply_delete(original, 5, 1)
after_both_2 = apply_insert(after_bob, 5, ", world")
print(f"Order 2 (Bob→Alice): '{original}' → '{after_bob}' → '{after_both_2}'")
print(f"  ✅ Happens to work, but only by luck of ordering!")
print()
print("💡 We need a way to adjust operations so they work regardless of order.")

## 🔧 What Is an Operation?

In OT, every edit to a document is represented as an **operation**:

| Operation | Fields | Example | Meaning |
|-----------|--------|---------|---------|
| **INSERT** | position, content | INSERT(5, ", world") | Insert ", world" at position 5 |
| **DELETE** | position, length | DELETE(5, 1) | Delete 1 character at position 5 |

Every operation is **contextual** — it was created while looking at a specific version of the document. When two users create operations based on the same version, those operations need to be **transformed** before one can be applied after the other.

In [ ]:
# Let's define operations as simple dictionaries

def make_insert(position, content, site=0):
    """Create an INSERT operation.

    `site` is a stable per-user id. Two users can legitimately insert at the
    exact same position; `site` is what lets every replica agree on which of
    the two goes first. (See the tie-break rule below.)
    """
    return {"type": "insert", "position": position, "content": content, "site": site}

def make_delete(position, length=1, site=0):
    """Create a DELETE operation covering the half-open range [position, position+length)."""
    return {"type": "delete", "position": position, "length": length, "site": site}

def apply_op(text, op):
    """Apply an operation to a text string."""
    if op["type"] == "insert":
        return apply_insert(text, op["position"], op["content"])
    elif op["type"] == "delete":
        return apply_delete(text, op["position"], op["length"])
    return text

# Demo: operations on a simple document
doc = ""
print(f"Start: '{doc}'")

ops = [
    make_insert(0, "Hello"),
    make_insert(5, "!"),
    make_insert(5, ", world"),
    make_delete(12, 1),  # remove the "!"
]

for i, op in enumerate(ops):
    doc = apply_op(doc, op)
    print(f"  Op {i+1}: {op['type'].upper()}({op['position']}, {op.get('content', op.get('length', ''))}) → '{doc}'")

print(f"\nFinal: '{doc}'")

## 🔄 The OT Algorithm: Transform Before Applying

The core idea of OT is simple:

> **Before applying an operation, transform it against all operations that have been applied since it was created.**

The transformation rules are intuitive:

### Rule 1: INSERT before INSERT
If operation A **inserts** text before operation B's position, B's position shifts **right** by the length of A's insertion.

### Rule 2: DELETE before INSERT
If operation A **deletes** text before operation B's position, B's position shifts **left** by the length of A's deletion.

### Rule 3: INSERT before DELETE
If operation A **inserts** text before operation B's delete position, B's position shifts **right**.

### Rule 4: DELETE before DELETE
If operation A **deletes** text before operation B's delete position, B's position shifts **left**.

```
Think of it like editing a numbered list:
If someone adds an item above yours, your item's number increases.
If someone removes an item above yours, your item's number decreases.
```

Those four rules are the ones everybody writes first. They are also not enough —
three cases need more than a position shift, and each one is a bug that a lab
which only tests the happy path will never notice:

### Rule 5: two INSERTs at *exactly* the same position
Shifting "B right if B is at or after A" is not symmetric: transform it the other
way round and A shifts right too, so the two users end up with the words in
opposite order. The tie has to be broken by something both sides agree on — the
**site id**.

### Rule 6: overlapping DELETEs
If A already deleted some of the characters B wants to delete, B must delete
**only the survivors**. Shifting B's position without shrinking its length makes
B eat `length` characters that nobody asked to remove. Two users pressing
Backspace on the same word is enough to trigger it.

### Rule 7: an INSERT *inside* a DELETE range
Inserts always survive. If A typed in the middle of the text B is deleting,
B has to **split into two deletes**, one on each side of the new text.

Rules 5–7 are why `transform()` below returns a **list** of operations:
usually one, sometimes two (a split), sometimes none (B was already deleted).

In [ ]:
def transform(op_a, op_b):
    """
    Transform operation B against operation A.

    Precondition: both ops were created against the same document state.
    A has already been applied. We need to adjust B so it still expresses
    the same intent on the post-A document.

    Returns a LIST of operations, to be applied in the order given:
      * 1 op  — the normal case (a shifted position)
      * 2 ops — A inserted inside the range B deletes, so B splits (Rule 7)
      * 0 ops — A already deleted everything B wanted to delete (Rule 6)
    """
    if op_b["type"] == "insert":
        return [_transform_insert(op_a, op_b)]
    return _transform_delete(op_a, op_b)


def _transform_insert(op_a, op_b):
    """B is an INSERT. Inserts always survive — only the position moves."""
    b = dict(op_b)

    if op_a["type"] == "insert":
        if b["position"] > op_a["position"]:
            b["position"] += len(op_a["content"])          # Rule 1
        elif b["position"] == op_a["position"]:
            # Rule 5 — tie-break. Whoever has the lower site id keeps the
            # earlier slot, on BOTH replicas. Using ">= position" here instead
            # (the obvious version) makes A-then-B and B-then-A disagree.
            if op_a.get("site", 0) <= b.get("site", 0):
                b["position"] += len(op_a["content"])
    else:
        a_start = op_a["position"]
        a_end = a_start + op_a.get("length", 1)
        if b["position"] >= a_end:
            b["position"] -= (a_end - a_start)             # Rule 2
        elif b["position"] > a_start:
            b["position"] = a_start                        # typed inside the hole

    return b


def _transform_delete(op_a, op_b):
    """B is a DELETE of the half-open range [start, end). Returns 0, 1 or 2 ops."""
    site = op_b.get("site", 0)
    b_start = op_b["position"]
    b_end = b_start + op_b.get("length", 1)

    if op_a["type"] == "insert":
        p = op_a["position"]
        ins_len = len(op_a["content"])
        if p <= b_start:
            return [make_delete(b_start + ins_len, b_end - b_start, site=site)]   # Rule 3
        if p >= b_end:
            return [dict(op_b)]                                                    # untouched
        # Rule 7: A typed inside the range B deletes. B must not swallow the new
        # text, so it splits around it. The RIGHT half is returned first — both
        # positions are in the same coordinate frame, and deleting the right
        # range first leaves the left range's position still valid.
        return [
            make_delete(p + ins_len, b_end - p, site=site),
            make_delete(b_start, p - b_start, site=site),
        ]

    # Rule 6: A is also a DELETE. B must only remove what is left of its range.
    a_start = op_a["position"]
    a_end = a_start + op_a.get("length", 1)
    overlap = max(0, min(a_end, b_end) - max(a_start, b_start))
    remaining = (b_end - b_start) - overlap
    if remaining <= 0:
        return []                       # A already deleted all of it
    # shift left by however much A removed *before* where B now starts
    new_start = b_start - max(0, min(a_end, b_start) - a_start)
    return [make_delete(new_start, remaining, site=site)]


def apply_ops(text, ops):
    """Apply a list of operations, in order."""
    for op in ops:
        text = apply_op(text, op)
    return text


def describe(op):
    """Short human-readable form of an op, e.g. INSERT(5, ', world')."""
    arg = repr(op["content"]) if op["type"] == "insert" else op.get("length", 1)
    return f"{op['type'].upper()}({op['position']}, {arg})"


print("OT Transform function defined! ✅")
print("Let's test it on our Alice & Bob example...")

In [ ]:
# Back to our example: Alice and Bob both edit "Hello!"
original = "Hello!"

alice_op = make_insert(5, ", world", site=1)  # insert ", world" at position 5
bob_op = make_delete(5, 1, site=2)            # delete 1 char at position 5 (the "!")

print(f"Original: '{original}'")
print(f"Alice: {describe(alice_op)}")
print(f"Bob:   {describe(bob_op)}")
print()

# The server receives Alice's op first, applies it
server_doc = apply_op(original, alice_op)
print(f"Server applies Alice's op: '{server_doc}'")

# Now the server needs to apply Bob's op — but it was created against the OLD document!
# We transform Bob's op against Alice's op
bob_transformed = transform(alice_op, bob_op)
print(f"\nBob's original op:     {describe(bob_op)}")
print(f"Bob's transformed op:  {' + '.join(describe(t) for t in bob_transformed)}")
print(f"  (shifted right by {len(alice_op['content'])} because Alice inserted {len(alice_op['content'])} chars before position 5)")

# Apply the transformed op
final = apply_ops(server_doc, bob_transformed)
print(f"\nFinal document: '{final}'")

# The same thing must happen if the server had seen Bob first — that symmetry
# is the property OT is judged on, so assert it rather than eyeball it.
final_other_order = apply_ops(apply_op(original, bob_op), transform(bob_op, alice_op))
assert final == final_other_order == "Hello, world", (
    f"OT diverged: Alice-first gave {final!r}, Bob-first gave {final_other_order!r}"
)
print(f"\n✅ Both intents preserved, in either arrival order: '{final_other_order}'")

## 🧪 More OT Examples

Let's test more scenarios to build intuition for how OT handles different cases.

In [ ]:
def simulate_concurrent_edit(original, op_a, op_b, name_a="User A", name_b="User B"):
    """
    Simulate two users making concurrent edits and show how OT resolves them.

    Both ops were created against the same `original` document. We apply op_a
    first, then transform and apply op_b — and then we do the whole thing the
    other way round and check the two documents match.

    That check has a name: **TP1** (transformation property 1). It is the
    contract every OT implementation has to satisfy:

        apply(apply(doc, a), transform(a, b)) == apply(apply(doc, b), transform(b, a))

    A transform that fails TP1 lets two users' screens drift apart silently,
    which is why we assert it here instead of reading the output.
    """
    print(f"Original: '{original}'")
    print(f"{name_a}: {describe(op_a)}")
    print(f"{name_b}: {describe(op_b)}")
    print()

    # Apply A first
    after_a = apply_op(original, op_a)
    print(f"After {name_a}: '{after_a}'")

    # Transform B against A, then apply
    transformed_b = transform(op_a, op_b)
    print(f"{name_b} transformed: {' + '.join(describe(t) for t in transformed_b) or '(dropped)'}")

    final = apply_ops(after_a, transformed_b)
    print(f"Final: '{final}'")

    # ...and now the other arrival order, which must land on the same text.
    other = apply_ops(apply_op(original, op_b), transform(op_b, op_a))
    assert final == other, f"TP1 violated: {name_a}-first={final!r} but {name_b}-first={other!r}"
    print(f"TP1 ✅  {name_b}-first gives the same text")
    print()
    return final


# Example 1: Two inserts at different positions
print("═" * 60)
print("Example 1: Two inserts at different positions")
print("═" * 60)
simulate_concurrent_edit(
    "Hello",
    make_insert(0, "Oh, ", site=1),      # User A adds "Oh, " at the start
    make_insert(5, " world", site=2),    # User B adds " world" at the end
    "Alice", "Bob"
)

# Example 2: Two inserts at the SAME position
print("═" * 60)
print("Example 2: Two inserts at the same position (Rule 5 — site tie-break)")
print("═" * 60)
simulate_concurrent_edit(
    "Hello world",
    make_insert(5, " beautiful", site=1),  # Alice adds "beautiful" after "Hello"
    make_insert(5, " wonderful", site=2),  # Bob also adds at position 5
    "Alice", "Bob"
)

# Example 3: Delete then insert
print("═" * 60)
print("Example 3: Delete before insert")
print("═" * 60)
simulate_concurrent_edit(
    "Hello world!",
    make_delete(5, 6, site=1),            # Alice deletes " world" (6 chars at position 5)
    make_insert(12, " :)", site=2),       # Bob adds smiley at the end
    "Alice", "Bob"
)

# Example 4: Both users delete the SAME word (Rule 6)
print("═" * 60)
print("Example 4: Overlapping deletes — nobody asked for extra characters to go")
print("═" * 60)
result = simulate_concurrent_edit(
    "Hello cruel world",
    make_delete(6, 6, site=1),            # Alice selects "cruel " and hits Backspace
    make_delete(6, 6, site=2),            # Bob does exactly the same thing
    "Alice", "Bob"
)
assert result == "Hello world", f"overlapping deletes ate the wrong characters: {result!r}"

# Example 5: Alice types inside the run of text Bob is deleting (Rule 7)
print("═" * 60)
print("Example 5: Insert inside a deleted range — the delete splits in two")
print("═" * 60)
result = simulate_concurrent_edit(
    "Hello cruel world",
    make_insert(9, "!!", site=1),         # Alice types "!!" in the middle of "cruel"
    make_delete(6, 6, site=2),            # Bob deletes "cruel "
    "Alice", "Bob"
)
assert result == "Hello !!world", f"insert was swallowed by the concurrent delete: {result!r}"

## 🧪 Does It Actually Converge? (TP1)

Five worked examples is five examples. The property we actually need is that
**every** pair of concurrent operations converges, and the only way to know is
to check them all.

The document below is 8 characters, which makes the space of "one insert or one
delete, anywhere" small enough to test **exhaustively** — every pair, both
arrival orders. We run the same sweep against the naive four-rule transform
(the one from the top of this notebook) so you can see exactly how much the
extra rules buy.

In [ ]:
# Exhaustive TP1 check: for every pair of concurrent ops, does arrival order matter?
import itertools

def naive_transform(op_a, op_b):
    """The transform almost everyone writes first: Rules 1-4 only, position
    shifts and nothing else. It looks right. It is wrong in three places."""
    b = dict(op_b)
    if op_a["type"] == "insert":
        if b["position"] >= op_a["position"]:
            b["position"] += len(op_a["content"])
    else:
        if b["position"] > op_a["position"]:
            b["position"] = max(op_a["position"], b["position"] - op_a["length"])
    return [b]


DOC = "abcdefgh"

def with_site(op, site):
    op = dict(op)
    op["site"] = site
    return op

# every single-op edit that fits in DOC
candidates = (
    [make_insert(p, "XY") for p in range(len(DOC) + 1)]
    + [make_delete(p, n) for p in range(len(DOC)) for n in (1, 3) if p + n <= len(DOC)]
)

def tp1_failures(tf):
    """Pairs (a, b) where applying a-then-b and b-then-a end up different."""
    bad = []
    for op_a, op_b in itertools.product(candidates, repeat=2):
        a, b = with_site(op_a, 1), with_site(op_b, 2)   # two different users
        if apply_ops(apply_op(DOC, a), tf(a, b)) != apply_ops(apply_op(DOC, b), tf(b, a)):
            bad.append((a, b))
    return bad

naive_bad = tp1_failures(naive_transform)
ours_bad = tp1_failures(transform)
total = len(candidates) ** 2

print(f"Checked {total} concurrent op pairs on '{DOC}' (both arrival orders each)")
print(f"  Rules 1-4 only:  {len(naive_bad):>4} pairs diverge  ❌")
print(f"  Rules 1-7:       {len(ours_bad):>4} pairs diverge  ✅")
print()
print("A sample of what the naive transform gets wrong:")
for a, b in naive_bad[:3]:
    ab = apply_ops(apply_op(DOC, a), naive_transform(a, b))
    ba = apply_ops(apply_op(DOC, b), naive_transform(b, a))
    print(f"  {describe(a):<20} vs {describe(b):<20} → '{ab}' or '{ba}'")

assert not ours_bad, f"our transform violates TP1 on {len(ours_bad)} pairs, e.g. {ours_bad[0]}"
assert naive_bad, "the naive transform is supposed to be broken — this demo has gone degenerate"

# Converging is necessary but not sufficient. A transform can agree with itself
# and still destroy text: two users hitting Backspace on the same three
# characters must remove three characters, not six.
a = make_delete(2, 3, site=1)
b = make_delete(2, 3, site=2)
naive_result = apply_ops(apply_op(DOC, a), naive_transform(a, b))
our_result = apply_ops(apply_op(DOC, a), transform(a, b))
print()
print(f"Both users delete '{DOC[2:5]}' from '{DOC}':")
print(f"  Rules 1-4 only:  '{naive_result}'  ← ate {len(DOC) - 3 - len(naive_result)} characters nobody deleted ❌")
print(f"  Rules 1-7:       '{our_result}'  ✅")

assert our_result == "abfgh", f"overlapping deletes are wrong: {our_result!r}"
assert len(naive_result) < len(our_result), "the over-deletion bug no longer reproduces — fix the prose"

## 📝 Seeing OT in the Database

Our doc server stores every operation in PostgreSQL. Let's look at the operations that built our seed documents.

In [ ]:
# Query the operations table
conn = get_db()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cursor.execute("""
    SELECT o.id, o.op_type, o.position, o.content, o.length,
           u.display_name AS user_name
    FROM operations o
    JOIN users u ON o.user_id = u.id
    WHERE o.document_id = 1 AND o.version = 0
    ORDER BY o.id
""")
ops = cursor.fetchall()
conn.close()

# version = 0 selects the ops that snapshot v1 compacted -- i.e. the ones that
# built the seed document. Notebook 3 appends more ops (at version 1 and up);
# leaving them in would mean replaying edits the v1 snapshot never saw.
print("📜 Operations log for Document 1 (Meeting Notes), up to snapshot v1:")
print(f"{'ID':>3}  {'User':<16} {'Type':<8} {'Pos':>4}  Content")
print("-" * 70)
for op in ops:
    content = op['content'][:40] + '...' if len(op['content'] or '') > 40 else op['content']
    print(f"{op['id']:>3}  {op['user_name']:<16} {op['op_type']:<8} {op['position']:>4}  {content}")

print(f"\n💡 Notice how Alice, Bob, and Charlie all contributed operations!")
print(f"   Each operation was applied sequentially by the server using OT.")

In [ ]:
# Let's replay the operations to rebuild the document from scratch

doc = ""  # start empty
print("🔄 Replaying operations from the database:")
print()

for op in ops:
    op_dict = {
        "type": op["op_type"],
        "position": op["position"],
        "content": op["content"] or "",
        "length": op["length"] or 0,
    }
    doc = apply_op(doc, op_dict)
    preview = doc[:60] + '...' if len(doc) > 60 else doc
    print(f"  Op {op['id']:>2} by {op['user_name']:<10} → '{preview}'")

print(f"\n📄 Final document ({len(doc)} chars):")
print(doc)

# The operations log is only useful if replaying it reproduces the document.
# Snapshot v1 is what the server believes doc 1 contains, so the two must match
# character for character — if they ever stop matching, the log has rotted.
conn = get_db()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cursor.execute("SELECT content FROM snapshots WHERE document_id = 1 AND version = 1")
snapshot_text = cursor.fetchone()["content"]
conn.close()

assert doc == snapshot_text, (
    f"replaying the op log gave {len(doc)} chars but snapshot v1 has "
    f"{len(snapshot_text)} — the operations table does not rebuild the document.\n"
    f"  replay:   {doc[:70]!r}\n  snapshot: {snapshot_text[:70]!r}"
)
print("\n✅ Replayed log matches snapshot v1 exactly.")

## 🏗️ How the Server Uses OT

Here's the flow when two users edit simultaneously:

```
  Alice (Client A)              Server                  Bob (Client B)
       │                          │                          │
       │──INSERT(5, ", world")──►│                          │
       │                          │  1. Apply Alice's op     │
       │                          │  2. Store in operations  │
       │◄────── ACK ─────────────│                          │
       │                          │──broadcast to Bob───────►│
       │                          │                          │
       │                          │◄──DELETE(5, 1)──────────│
       │                          │  3. Transform Bob's op   │
       │                          │     against Alice's op   │
       │                          │  4. DELETE(5,1) becomes  │
       │                          │     DELETE(12,1)         │
       │                          │  5. Apply transformed op │
       │                          │  6. Store in operations  │
       │                          │──────── ACK ────────────►│
       │◄──broadcast to Alice────│                          │
       │                          │                          │
```

**Key insight**: The server is the single source of truth. All operations go through it, and it applies OT to maintain consistency.

In [ ]:
# Let's simulate the full server-side OT flow

class SimpleOTServer:
    """A minimal OT server that processes operations sequentially."""

    def __init__(self, initial_text=""):
        self.text = initial_text
        self.history = []  # every op actually applied, in application order
        self.version = 0   # == len(self.history) — how many ops the doc has seen

    def receive_op(self, op, client_version, user_name="Unknown", verbose=True):
        """
        Receive an operation from a client.

        client_version: how many ops the client had seen when it built this op.
        Everything in history after that index happened concurrently, so the op
        is transformed against each of them in turn — in application order,
        because each transform moves the op one document version forward.
        """
        pending = [dict(op)]
        for past_op in self.history[client_version:]:
            pending = [t for p in pending for t in transform(past_op, p)]
            # A split delete leaves two ops in the SAME coordinate frame;
            # applying the rightmost first keeps the other's position valid.
            pending.sort(key=lambda o: -o["position"])

        old_text = self.text
        for t in pending:
            self.text = apply_op(self.text, t)
            self.history.append(t)
        self.version = len(self.history)

        if verbose:
            def sig(o):
                return (o["type"], o["position"], o.get("content", ""), o.get("length", 0))
            print(f"  [{user_name}] version {client_version} → {self.version}")
            if [sig(t) for t in pending] != [sig(op)]:
                after = " + ".join(describe(t) for t in pending) or "(dropped — already deleted)"
                print(f"    ⚡ Transformed: {describe(op)} → {after}")
            print(f"    Document: '{self.text}'")

        return pending


# Simulate concurrent editing
print("🖥️  Simulating OT Server")
print("=" * 50)

server = SimpleOTServer("Hello!")
print(f"Initial document: '{server.text}' (version {server.version})")
print()

# Both Alice and Bob see "Hello!" at version 0
# Alice types ", world" at position 5 (version 0)
# Bob deletes "!" at position 5 (also version 0)

print("Alice sends INSERT(5, ', world') based on version 0:")
server.receive_op(make_insert(5, ", world", site=1), client_version=0, user_name="Alice")

print("\nBob sends DELETE(5, 1) based on version 0:")
server.receive_op(make_delete(5, 1, site=2), client_version=0, user_name="Bob")

print(f"\n✅ Final document: '{server.text}'")
print(f"   Both intents preserved! Alice's comma and Bob's deletion both applied correctly.")

assert server.text == "Hello, world", f"server OT produced {server.text!r}"

In [ ]:
# A more complex scenario: three users editing simultaneously

print("Three-User OT Simulation")
print("=" * 50)

# Starting text positions:  "The quick brown fox"
#                            0    5   9     15 19
# All three users see version 0 and edit concurrently. Each gets a site id,
# which is what breaks the tie between Alice's and Bob's inserts at position 19.
concurrent_ops = [
    (make_insert(19, " and lazy dog", site=1), "Alice"),
    (make_insert(19, " jumps", site=2), "Bob"),
    (make_delete(4, 6, site=3), "Charlie"),
]

server = SimpleOTServer("The quick brown fox")
print(f"Initial: '{server.text}' (version {server.version})")
print()

for op, who in concurrent_ops:
    print(f"{who} sends {describe(op)} (version 0):")
    server.receive_op(op, client_version=0, user_name=who)
    print()

print(f"Final: '{server.text}'")
print()
print("All three concurrent edits were transformed and applied correctly.")
print("Notice how Bob's INSERT(19) became INSERT(32) - shifted by Alice's 13-char insert.")
print("And Charlie's delete at position 4 still removed 'quick ' even after the other edits.")

assert server.text == "The brown fox and lazy dog jumps", f"got {server.text!r}"

# The server is allowed to receive those three ops in any order. Whatever order
# it picks, the document it ends up with must be the same one — otherwise
# "the server is the source of truth" would depend on network timing.
import itertools

outcomes = {}
for perm in itertools.permutations(concurrent_ops):
    replay = SimpleOTServer("The quick brown fox")
    for op, who in perm:
        replay.receive_op(op, client_version=0, user_name=who, verbose=False)
    outcomes.setdefault(replay.text, []).append(" → ".join(w for _, w in perm))

print()
print(f"Replayed all {sum(len(v) for v in outcomes.values())} arrival orders:")
for text, orders in outcomes.items():
    print(f"  '{text}'  ({len(orders)} orders)")

assert len(outcomes) == 1, f"arrival order changed the document: {list(outcomes)}"

## ⚠️ Where This Transform Stops Being Enough (TP2)

TP1 — the property we just verified exhaustively — covers **two** concurrent
operations. Real OT has a second property, **TP2**:

> transforming an op against two concurrent ops must give the same result
> whichever of the two you transform against first.

TP2 is much harder, and our transform does **not** satisfy it. The failure needs
at least three concurrent ops and a delete that pulls two inserts onto the same
position — at which point the site tie-break is being asked about a tie that
did not exist when the ops were created.

This is not an oversight to paper over; it is the reason OT systems are built
the way they are. **The server imposes a total order.** Every client transforms
against the server's history in the server's order, so no client ever has to
resolve the ambiguity below on its own. Google Wave's OT was famously hard
precisely because it tried to be correct without that crutch.

In [ ]:
# A TP2 counterexample: four concurrent ops, two possible answers.
ops = [
    (make_insert(5, "X", site=1), "Alice"),
    (make_delete(3, 2, site=2), "Bob"),
    (make_delete(2, 2, site=3), "Carol"),
    (make_insert(2, "PQR", site=4), "Dave"),
]

outcomes = {}
for perm in itertools.permutations(ops):
    replay = SimpleOTServer("abcdefgh")
    for op, who in perm:
        replay.receive_op(op, client_version=0, user_name=who, verbose=False)
    outcomes.setdefault(replay.text, []).append(" → ".join(w for _, w in perm))

print("Four users edit 'abcdefgh' at the same time, from version 0:")
for op, who in ops:
    print(f"  {who:<6} {describe(op)}")
print()
print(f"Replaying all {sum(len(v) for v in outcomes.values())} arrival orders gives:")
for text, orders in sorted(outcomes.items()):
    print(f"  '{text}'  ({len(orders)} orders, e.g. {orders[0]})")
print()
print("Both answers keep every character and every insertion — they only disagree")
print("on whether 'X' lands before or after 'PQR'. Deletes collapsed two positions")
print("that started out different, and nothing in the ops records which came first.")
print()
print("💡 The fix is not a cleverer transform, it is the architecture: one server")
print("   decides the order, everyone else replays it. That is the trade-off in the")
print("   table below — OT buys low memory and pays with a single point of ordering.")

assert len(outcomes) > 1, (
    "the TP2 counterexample stopped diverging — either transform() changed or the "
    "prose above is now wrong; do not leave the claim standing untested"
)

## ⚠️ Challenges with OT

OT is powerful but comes with important trade-offs:

| Challenge | Description |
|-----------|-------------|
| **Central server required** | All operations must flow through a single server that maintains ordering |
| **Complex to implement correctly** | Ties, overlapping deletes and delete-splitting are all silent-corruption bugs |
| **TP2 needs a total order** | Three-way concurrency is ambiguous without a server to decide the order |
| **Scaling limit** | One document = one server (Google Docs limits to ~100 concurrent editors) |
| **Latency sensitive** | Operations should be acknowledged within ~100ms for a good UX |

### What this notebook's transform does and does not do

- ✅ **TP1**, verified exhaustively over every pair of single-character-range ops
- ✅ Overlapping deletes never remove text neither user selected
- ✅ Inserts are never swallowed by a concurrent delete (the delete splits)
- ❌ **TP2** — three or more concurrent ops can be ordered two ways (shown above)
- ❌ Rich text (bold, tables, images) — real OT transforms attribute ops too
- ❌ Cursor transformation, undo, and client-side pending queues (Notebook 3)

### Why Google Docs Chose OT

Despite these challenges, OT has important advantages:

- **Low memory**: only needs the current document + recent operations (not the full history)
- **Well-suited for text**: insert/delete operations map naturally to text editing
- **Battle-tested**: Google has used OT since 2006 (Google Wave → Google Docs)
- **100 editors is enough**: most real documents have 2-10 concurrent editors

## 🧹 Cleanup

In [ ]:
# Nothing to clean up — we only read from the database in this notebook
print("🧹 No cleanup needed — all operations were in-memory simulations.")

## 📚 Summary

### Key Takeaways

1. **Concurrent edits break documents** — applying the same operations in different orders gives different results
2. **OT transforms operations** — adjusting positions so each edit preserves the user's original intent
3. **The server is the authority** — all operations go through a central server that enforces ordering
4. **Operations are contextual** — each op was created against a specific document version
5. **Google Docs uses OT** — it's low memory, fast, and works great for ≤100 editors

### Next Up

In **Notebook 2**, we'll explore **CRDTs** — an alternative approach where operations can be applied in **any order** without a central server. This is what Figma and Apple Notes use.